# Evaluate baseline and fine-tuned CAMS rollouts

This notebook compares baseline CAMS rollouts and fine-tuned CAMS rollouts
with the ground-truth CAMS test dataset. It discovers the data layout,
matches forecasts using

\[
\text{initialization time} + \text{lead time} = \text{ground-truth valid time},
\]

validates coordinates before computing metrics, writes tidy CSV and
NetCDF results, creates a Markdown summary, and saves comparison figures.

**Important alignment contract**

- No interpolation, resizing, positional truncation, or temporal
  nearest-neighbor matching is performed.
- Longitude values are converted to `[-180, 180)` and coordinates are
  sorted. Numerically equivalent coordinates may be snapped within the
  configured tolerance; data values are never interpolated.
- `spatial_alignment` defaults to `coordinate_intersection` when the training
  YAML declares `data.patch_alignment_strategy: crop`, and to `exact` otherwise.
  If grids differ only because a rollout explicitly crops boundary coordinates,
  use `spatial_alignment: coordinate_intersection`. That selection uses the
  coordinate-matched intersection and records a warning in the summary.
- Ensemble rollouts are reduced to their ensemble mean by default. This can
  be changed with `ensemble_reduction`.


## 1. Configuration

The only required notebook input is a path to a YAML file. Set `CONFIG_PATH`
in the tagged parameter cell below. Notebook runners such as Papermill can
inject it with `-p CONFIG_PATH path/to/config.yaml`. The
`CAMS_EVALUATION_CONFIG` environment variable is also supported.

Any training YAML with a non-empty top-level `case_name` works; an
`evaluation` section is optional. When it is absent, paths are derived from
the training YAML's `paths` section. If `paths.data_case_name` is set, it
selects the prepared-data folder independently of the experiment name. Lead hours are derived from
`data.target_lead_times × rollout.rollout_step_hours` when both settings are
available, otherwise they default to `[12, 24, 36, 48, 60, 72]`:

- truth: `{paths.data_dir}/{paths.data_case_name or case_name}/test.nc`
- baseline: the first existing `outputs/cams_rollouts` or
  `examples/outputs/cams_rollouts` directory
- fine-tuned: `{paths.output_dir}/{case_name}`
- results: `{paths.output_dir}/{case_name}/evaluation`

Relative `evaluation` path overrides are resolved from the repository root.
A supported evaluation section is:

```yaml
case_name: example_case

evaluation:
  ground_truth_path: null
  baseline_rollout_path: null
  finetuned_rollout_path: null
  output_dir: null

  lead_times_hours: [12, 24, 36, 48, 60, 72]
  target_variables: null
  levels: null
  metrics: [bias, mae, rmse, spatial_correlation]

  spatial_alignment: exact
  coordinate_tolerance: 2.0e-5
  level_tolerance: 1.0e-3
  time_tolerance_seconds: 1.0
  lead_time_tolerance_hours: 1.0e-6

  ensemble_reduction: mean
  generate_comparison_figures: true
  percentage_epsilon: 1.0e-12
  save_per_case_metrics: true
  recursive_file_search: false
  file_pattern: "*.nc"
  extra_dim_indexers: {}

  generate_maps: false
  map_lead_times_hours: [24, 48, 72]
  map_variables: null
  map_levels: null
  map_initialization_times: null
  max_map_forecasts_per_selection: 1
```

`target_variables` accepts strings or the existing structured entries with
`dataset_name`, `aurora_name`, `kind`, and optional `loss_levels`.


In [1]:
# YAML file to evaluate. Use an absolute path or a path relative to the repository root.
CONFIG_PATH = '/data/aurora/finetune/aurora_NO2_finetune_US-WEST_3day_lead_flow_matching_config.yaml'


In [2]:
from __future__ import annotations

import glob
import math
import os
import re
import warnings
from collections import Counter, OrderedDict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping, Sequence

import numpy as np
import pandas as pd
import xarray as xr
import yaml
from tqdm import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Main notebook input. A notebook runner can inject CONFIG_PATH after the
# tagged parameter cell; the environment variable is a convenient fallback.
CONFIG_PATH = globals().get("CONFIG_PATH") or os.environ.get(
    "CAMS_EVALUATION_CONFIG"
)
if CONFIG_PATH is None or not str(CONFIG_PATH).strip():
    raise ValueError(
        "Set CONFIG_PATH in the parameter cell, inject it with a notebook "
        "runner, or export CAMS_EVALUATION_CONFIG. Any training YAML with a "
        "top-level case_name is supported; an evaluation section is optional."
    )

CONFIG_PATH = Path(CONFIG_PATH).expanduser()
if not CONFIG_PATH.is_absolute():
    cwd_candidate = (Path.cwd() / CONFIG_PATH).resolve()
    repository_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / ".git").exists()
        ),
        Path.cwd(),
    )
    root_candidate = (repository_root / CONFIG_PATH).resolve()
    CONFIG_PATH = cwd_candidate if cwd_candidate.is_file() else root_candidate
else:
    CONFIG_PATH = CONFIG_PATH.resolve()
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"YAML configuration file does not exist: {CONFIG_PATH}")


In [3]:
REQUIRED_METRICS = ("bias", "mae", "rmse", "spatial_correlation")
DEFAULT_LEAD_HOURS = [12, 24, 36, 48, 60, 72]


class EvaluationError(RuntimeError):
    """Raised when evaluation inputs cannot be matched without guessing."""


@dataclass(frozen=True)
class VariableSpec:
    """A target variable and its possible names in input datasets."""

    dataset_name: str
    aliases: tuple[str, ...]
    kind: str | None = None
    loss_levels: tuple[float, ...] | None = None


@dataclass
class WarningLog:
    """Deduplicated warning messages with occurrence counts."""

    counts: Counter[str] = field(default_factory=Counter)

    def add(self, message: str) -> None:
        self.counts[str(message)] += 1

    def messages(self) -> list[str]:
        return [
            f"{message}" + (f" (repeated {count} times)" if count > 1 else "")
            for message, count in self.counts.items()
        ]


def find_repository_root(start: Path) -> Path:
    """Find the nearest parent containing `.git`, falling back to the CWD."""
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return cwd


def resolve_from_root(value: str | Path, repository_root: Path) -> Path:
    """Resolve a path relative to the repository root."""
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = repository_root / path
    return path.resolve()


def resolve_from_base(value: str | Path, base: Path) -> Path:
    """Resolve a path relative to a YAML-defined project directory."""
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = base / path
    return path.resolve()


def load_configuration(config_path: Path) -> tuple[dict[str, Any], dict[str, Any]]:
    """Load YAML and derive backward-compatible evaluation settings."""
    raw = yaml.safe_load(config_path.read_text())
    if not isinstance(raw, dict):
        raise ValueError(f"YAML root must be a mapping: {config_path}")

    case_name = str(raw.get("case_name", "")).strip()
    if not case_name:
        raise ValueError(
            f"Configuration {config_path} must define a non-empty top-level `case_name`."
        )

    repository_root = find_repository_root(config_path.parent)
    paths_config = raw.get("paths", {}) or {}
    if not isinstance(paths_config, dict):
        raise TypeError("The YAML paths value must be a mapping or null.")

    training_project_root = Path(
        paths_config.get("project_root", config_path.parent)
    ).expanduser()
    if not training_project_root.is_absolute():
        training_project_root = (
            config_path.parent / training_project_root
        ).resolve()
    else:
        training_project_root = training_project_root.resolve()

    data_dir = resolve_from_base(
        paths_config.get("data_dir", training_project_root / "data"),
        training_project_root,
    )
    data_case_name = str(paths_config.get("data_case_name") or case_name).strip()
    case_data_dir = (
        data_dir if data_dir.name == data_case_name else data_dir / data_case_name
    )
    training_output_base = resolve_from_base(
        paths_config.get("output_dir", training_project_root / "outputs"),
        training_project_root,
    )
    case_output_dir = (
        training_output_base
        if training_output_base.name == case_name
        else training_output_base / case_name
    )
    baseline_candidates = (
        repository_root / "outputs" / "cams_rollouts",
        repository_root / "examples" / "outputs" / "cams_rollouts",
    )
    baseline_default = next(
        (candidate for candidate in baseline_candidates if candidate.exists()),
        baseline_candidates[0],
    )

    supplied = raw.get("evaluation") or {}
    if not isinstance(supplied, dict):
        raise TypeError("The YAML `evaluation` value must be a mapping or null.")

    data_config = raw.get("data", {}) or {}
    rollout_config = raw.get("rollout", {}) or {}
    configured_leads = supplied.get("lead_times_hours")
    if configured_leads is None:
        lead_indices = data_config.get("target_lead_times")
        step_hours = rollout_config.get("rollout_step_hours")
        if lead_indices and step_hours is not None:
            configured_leads = [
                float(index) * float(step_hours) for index in lead_indices
            ]
        else:
            configured_leads = DEFAULT_LEAD_HOURS

    defaults: dict[str, Any] = {
        "lead_times_hours": configured_leads,
        "target_variables": None,
        "levels": None,
        "metrics": list(REQUIRED_METRICS),
        "spatial_alignment": (
            "coordinate_intersection"
            if str(data_config.get("patch_alignment_strategy", "")).lower() == "crop"
            else "exact"
        ),
        "coordinate_tolerance": 2.0e-5,
        "level_tolerance": 1.0e-3,
        "time_tolerance_seconds": 1.0,
        "lead_time_tolerance_hours": 1.0e-6,
        "ensemble_reduction": "mean",
        "generate_comparison_figures": True,
        "percentage_epsilon": 1.0e-12,
        "save_per_case_metrics": True,
        "recursive_file_search": False,
        "file_pattern": "*.nc",
        "extra_dim_indexers": {},
        "lead_time_numeric_units": "hours",
        "generate_maps": False,
        "generate_overall_maps": True,
        "map_lead_times_hours": [24, 48, 72],
        "map_variables": None,
        "map_levels": None,
        "map_initialization_times": None,
        "max_map_forecasts_per_selection": 1,
    }
    # YAML null means "use the derived/default value". This keeps the
    # documented evaluation block concise and backward compatible.
    non_null_supplied = {
        key: value for key, value in supplied.items() if value is not None
    }
    settings = {**defaults, **non_null_supplied}
    settings["case_name"] = case_name
    settings["repository_root"] = repository_root

    path_defaults = {
        "ground_truth_path": case_data_dir / "test.nc",
        "baseline_rollout_path": baseline_default,
        "finetuned_rollout_path": case_output_dir,
        "output_dir": case_output_dir / "evaluation",
    }
    for key, default in path_defaults.items():
        value = supplied.get(key)
        settings[key] = default.resolve() if value in (None, "") else resolve_from_root(
            value, repository_root
        )

    leads = [float(value) for value in settings["lead_times_hours"]]
    if not leads or any(not np.isfinite(value) or value < 0 for value in leads):
        raise ValueError("evaluation.lead_times_hours must contain finite non-negative values.")
    settings["lead_times_hours"] = list(OrderedDict.fromkeys(leads))

    mode = str(settings["spatial_alignment"]).lower()
    if mode not in {"exact", "coordinate_intersection"}:
        raise ValueError(
            "evaluation.spatial_alignment must be `exact` or `coordinate_intersection`."
        )
    settings["spatial_alignment"] = mode

    ensemble_reduction = str(settings["ensemble_reduction"]).lower()
    if ensemble_reduction not in {"mean", "median", "first"}:
        raise ValueError("evaluation.ensemble_reduction must be mean, median, or first.")
    settings["ensemble_reduction"] = ensemble_reduction

    configured_metrics = [str(value).lower() for value in settings.get("metrics") or []]
    omitted = [metric for metric in REQUIRED_METRICS if metric not in configured_metrics]
    if omitted:
        warnings.warn(
            "Required metrics were absent from evaluation.metrics and will still be "
            f"computed: {omitted}",
            stacklevel=2,
        )
    settings["metrics"] = list(OrderedDict.fromkeys(configured_metrics + list(REQUIRED_METRICS)))

    extra = {}
    extra.update(raw.get("data", {}).get("extra_dim_indexers", {}) or {})
    extra.update(settings.get("extra_dim_indexers", {}) or {})
    settings["extra_dim_indexers"] = extra
    return raw, settings


RAW_CONFIG, EVAL = load_configuration(CONFIG_PATH)
CASE_NAME = EVAL["case_name"]
OUTPUT_DIR = Path(EVAL["output_dir"])
FIGURE_DIR = OUTPUT_DIR / "figures"

print(f"case_name: {CASE_NAME}")
print(f"repository root: {EVAL['repository_root']}")
print(f"ground truth: {EVAL['ground_truth_path']}")
print(f"baseline rollouts: {EVAL['baseline_rollout_path']}")
print(f"fine-tuned rollouts: {EVAL['finetuned_rollout_path']}")
print(f"output directory: {OUTPUT_DIR}")
print(f"requested lead hours: {EVAL['lead_times_hours']}")


case_name: NO2_US-WEST_3day_lead_flow_matching
repository root: /data/aurora
ground truth: /data/aurora/data/NO2_US-WEST_3day_lead/test.nc
baseline rollouts: /data/aurora/examples/outputs/cams_rollouts
fine-tuned rollouts: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching
output directory: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation
requested lead hours: [12.0, 24.0, 36.0, 48.0, 60.0, 72.0]


## 2. Dataset discovery and forecast cataloging

The next cells use coordinate names, coordinate dtypes, CF-style units,
scalar attributes, and initialization timestamps embedded in filenames.
Filenames are only a fallback; records are always validated against
`valid_time = initialization_time + lead_time`.

A rollout path can be a NetCDF file, a directory, or a glob. Directory
search is top-level by default so prior `evaluation/` outputs cannot be
mistaken for rollouts. Set `recursive_file_search: true` only when needed.


In [4]:
LAT_NAMES = ("latitude", "lat", "nav_lat", "y")
LON_NAMES = ("longitude", "lon", "nav_lon", "x")
LEVEL_NAMES = (
    "level",
    "pressure_level",
    "pressure",
    "isobaricInhPa",
    "plev",
    "lev",
    "vertical",
    "altitude",
)
MEMBER_NAMES = ("member", "ensemble", "realization", "number")
INIT_NAMES = (
    "forecast_reference_time",
    "initialization_time",
    "forecast_initialization_time",
    "init_time",
    "reference_time",
)
VALID_NAMES = ("valid_time", "verification_time", "forecast_time")
LEAD_NAMES = (
    "lead_time",
    "leadtime",
    "forecast_period",
    "prediction_timedelta",
    "step",
    "horizon",
)
TIME_NAMES = ("time", "valid_time", "datetime", "date")


@dataclass(frozen=True)
class DatasetAxes:
    """Discovered coordinate/dimension names for one NetCDF file."""

    latitude: str
    longitude: str
    level: str | None
    member: str | None


@dataclass(frozen=True)
class ForecastRecord:
    """One forecast initialization and lead stored in a NetCDF file."""

    path: Path
    initialization_time: np.datetime64
    valid_time: np.datetime64
    lead_time_hours: float
    indexers: tuple[tuple[str, int], ...]

    @property
    def selector(self) -> dict[str, int]:
        return dict(self.indexers)


@dataclass
class ForecastCatalog:
    """Forecast records and per-file schemas for a rollout product."""

    label: str
    files: list[Path]
    records: list[ForecastRecord]
    axes: dict[Path, DatasetAxes]
    variables: dict[Path, set[str]]


def first_present(
    ds: xr.Dataset,
    names: Sequence[str],
    *,
    include_dims: bool = True,
    require_datetime: bool = False,
) -> str | None:
    """Return the first candidate present as a coordinate, variable, or dimension."""
    for name in names:
        if name in ds.coords or name in ds.variables or (include_dims and name in ds.dims):
            if require_datetime and name in ds:
                if not np.issubdtype(ds[name].dtype, np.datetime64):
                    continue
            return name
    return None


def discover_axes(ds: xr.Dataset, path: Path) -> DatasetAxes:
    """Discover one-dimensional latitude, longitude, level, and member axes."""
    data_cfg = RAW_CONFIG.get("data", {}) or {}
    lat_candidates = tuple(
        OrderedDict.fromkeys([str(data_cfg.get("lat_dim", "")), *LAT_NAMES])
    )
    lon_candidates = tuple(
        OrderedDict.fromkeys([str(data_cfg.get("lon_dim", "")), *LON_NAMES])
    )
    level_candidates = tuple(
        OrderedDict.fromkeys([str(data_cfg.get("level_dim", "")), *LEVEL_NAMES])
    )
    latitude = first_present(ds, [name for name in lat_candidates if name])
    longitude = first_present(ds, [name for name in lon_candidates if name])
    if latitude is None or longitude is None:
        raise EvaluationError(
            f"{path}: could not identify latitude/longitude coordinates. "
            f"Available dimensions={dict(ds.sizes)}, coordinates={list(ds.coords)}."
        )
    for label, name in (("latitude", latitude), ("longitude", longitude)):
        if name not in ds or ds[name].ndim != 1:
            raise EvaluationError(
                f"{path}: {label} coordinate `{name}` must be one-dimensional; "
                f"found dims={getattr(ds.get(name), 'dims', None)}."
            )
    return DatasetAxes(
        latitude=latitude,
        longitude=longitude,
        level=first_present(ds, [name for name in level_candidates if name]),
        member=first_present(ds, MEMBER_NAMES),
    )


def discover_files(source: Path) -> list[Path]:
    """Expand a rollout file, directory, or glob to a sorted NetCDF list."""
    source_text = str(source)
    has_magic = glob.has_magic(source_text)
    recursive = bool(EVAL["recursive_file_search"])
    pattern = str(EVAL["file_pattern"])

    if has_magic:
        paths = [Path(value).resolve() for value in glob.glob(source_text, recursive=recursive)]
    elif source.is_file():
        paths = [source.resolve()]
    elif source.is_dir():
        iterator = source.rglob(pattern) if recursive else source.glob(pattern)
        paths = [path.resolve() for path in iterator]
    else:
        raise FileNotFoundError(
            f"Rollout source does not exist and is not a matching glob: {source}"
        )

    files = sorted({path for path in paths if path.is_file() and path.suffix.lower() == ".nc"})
    if not files:
        raise FileNotFoundError(
            f"No NetCDF rollout files matched {source!s} "
            f"(pattern={pattern!r}, recursive={recursive})."
        )
    return files


def parse_datetime(value: Any, *, context: str) -> np.datetime64:
    """Parse one scalar datetime and normalize it to nanosecond resolution."""
    try:
        parsed = np.datetime64(value, "ns")
    except Exception as exc:
        raise EvaluationError(f"Could not parse {context} as datetime: {value!r}") from exc
    if np.isnat(parsed):
        raise EvaluationError(f"{context} is NaT.")
    return parsed


def initialization_from_filename(path: Path) -> np.datetime64 | None:
    """Extract an initialization timestamp from common rollout filenames."""
    patterns = (
        r"(?:init|initialization|forecast)[_-]?(\d{8}T\d{6})",
        r"(?:init|initialization|forecast)[_-]?(\d{14})",
        r"(?:^|_)rollout[_-](\d{8}_\d{6})(?:_|$)",
    )
    for pattern in patterns:
        match = re.search(pattern, path.stem, flags=re.IGNORECASE)
        if match:
            token = match.group(1).replace("_", "T")
            try:
                return np.datetime64(pd.to_datetime(token, format="%Y%m%dT%H%M%S"), "ns")
            except ValueError:
                try:
                    return np.datetime64(pd.to_datetime(token, format="%Y%m%d%H%M%S"), "ns")
                except ValueError:
                    pass
    return None


def lead_hours(da: xr.DataArray, path: Path) -> xr.DataArray:
    """Convert timedelta or numeric lead coordinates to floating-point hours."""
    if np.issubdtype(da.dtype, np.timedelta64):
        values = np.asarray(da.values) / np.timedelta64(1, "h")
        return xr.DataArray(values.astype(float), dims=da.dims, coords=da.coords)

    values = np.asarray(da.values, dtype=float)
    unit = str(da.attrs.get("units", "")).strip().lower()
    if not unit:
        unit = str(EVAL["lead_time_numeric_units"]).lower()
        DISCOVERY_WARNINGS.add(
            f"Numeric lead coordinate `{da.name}` has no units; interpreted as "
            f"{unit} by evaluation.lead_time_numeric_units."
        )
    if unit.startswith(("hour", "hr", "h")):
        factor = 1.0
    elif unit.startswith(("day", "d")):
        factor = 24.0
    elif unit.startswith(("minute", "min")):
        factor = 1.0 / 60.0
    elif unit.startswith(("second", "sec", "s")):
        factor = 1.0 / 3600.0
    else:
        raise EvaluationError(
            f"{path}: unsupported units {unit!r} for lead coordinate `{da.name}`. "
            "Use CF timedelta coordinates or set evaluation.lead_time_numeric_units."
        )
    return xr.DataArray(values * factor, dims=da.dims, coords=da.coords)


def as_data_array(ds: xr.Dataset, name: str | None) -> xr.DataArray | None:
    """Return a named coordinate/data variable, or None."""
    return None if name is None or name not in ds else ds[name]


def forecast_records_from_file(path: Path) -> tuple[list[ForecastRecord], DatasetAxes, set[str]]:
    """Inspect one rollout file and enumerate every initialization/lead record."""
    with xr.open_dataset(path, decode_times=True, decode_timedelta=True) as ds:
        axes = discover_axes(ds, path)
        lead_name = first_present(ds, LEAD_NAMES)
        lead_da = as_data_array(ds, lead_name)

        # Prefer explicit initialization metadata. `time` is an initialization
        # axis only when a separate lead coordinate is present.
        init_name = first_present(ds, INIT_NAMES)
        init_da = as_data_array(ds, init_name)
        scalar_init: np.datetime64 | None = None
        if init_da is None:
            for attr_name in INIT_NAMES:
                if attr_name in ds.attrs:
                    scalar_init = parse_datetime(
                        ds.attrs[attr_name], context=f"{path}:{attr_name}"
                    )
                    break
        if init_da is None and scalar_init is None:
            scalar_init = initialization_from_filename(path)
        if init_da is None and scalar_init is None and lead_da is not None:
            time_as_init = first_present(ds, ("time",), require_datetime=True)
            init_da = as_data_array(ds, time_as_init)

        # A datetime `time` coordinate is valid time for per-initialization files.
        valid_name = first_present(ds, VALID_NAMES, require_datetime=True)
        if valid_name is None and (scalar_init is not None or init_name is not None):
            candidate = first_present(ds, TIME_NAMES, require_datetime=True)
            if candidate != init_name:
                valid_name = candidate
        valid_da = as_data_array(ds, valid_name)

        if init_da is None and scalar_init is None:
            raise EvaluationError(
                f"{path}: could not determine forecast initialization time. Add an "
                "`initialization_time` attribute/coordinate, a CF "
                "`forecast_reference_time`, or an `init_YYYYMMDDTHHMMSS` filename."
            )
        if valid_da is None and lead_da is None:
            raise EvaluationError(
                f"{path}: could not determine valid time or lead time. Expected a "
                "datetime valid-time coordinate or a lead/step coordinate."
            )

        if init_da is None:
            init_da = xr.DataArray(scalar_init)
        elif not np.issubdtype(init_da.dtype, np.datetime64):
            try:
                init_da = xr.DataArray(
                    np.asarray(init_da.values).astype("datetime64[ns]"),
                    dims=init_da.dims,
                    coords=init_da.coords,
                )
            except Exception as exc:
                raise EvaluationError(
                    f"{path}: initialization coordinate `{init_da.name}` is not datetime-like."
                ) from exc

        lead_da_hours = lead_hours(lead_da, path) if lead_da is not None else None
        arrays = [init_da]
        if valid_da is not None:
            arrays.append(valid_da)
        if lead_da_hours is not None:
            arrays.append(lead_da_hours)
        broadcast = xr.broadcast(*arrays)
        init_b = broadcast[0]
        offset = 1
        valid_b = broadcast[offset] if valid_da is not None else None
        offset += int(valid_da is not None)
        lead_b = broadcast[offset] if lead_da_hours is not None else None

        init_values = np.asarray(init_b.values).astype("datetime64[ns]")
        if valid_b is None:
            assert lead_b is not None
            lead_values = np.asarray(lead_b.values, dtype=float)
            valid_values = init_values + np.rint(lead_values * 3_600_000_000_000).astype(
                "timedelta64[ns]"
            )
        else:
            valid_values = np.asarray(valid_b.values).astype("datetime64[ns]")
            inferred = (valid_values - init_values) / np.timedelta64(1, "h")
            if lead_b is None:
                lead_values = np.asarray(inferred, dtype=float)
            else:
                lead_values = np.asarray(lead_b.values, dtype=float)
                mismatch = np.abs(lead_values - np.asarray(inferred, dtype=float))
                tolerance = float(EVAL["lead_time_tolerance_hours"])
                if np.any(np.isfinite(mismatch) & (mismatch > tolerance)):
                    worst = float(np.nanmax(mismatch))
                    raise EvaluationError(
                        f"{path}: valid time disagrees with initialization + lead "
                        f"(maximum mismatch {worst:g} hours; tolerance {tolerance:g})."
                    )

        records: list[ForecastRecord] = []
        dims = init_b.dims
        shape = init_b.shape
        positions: Iterable[tuple[int, ...]] = np.ndindex(shape) if shape else [()]
        for position in positions:
            init_value = parse_datetime(init_values[position], context=f"{path}:initialization")
            valid_value = parse_datetime(valid_values[position], context=f"{path}:valid_time")
            lead_value = float(lead_values[position])
            if not np.isfinite(lead_value):
                raise EvaluationError(f"{path}: encountered a non-finite lead time.")
            selector = tuple((dim, int(index)) for dim, index in zip(dims, position))
            records.append(
                ForecastRecord(
                    path=path,
                    initialization_time=init_value,
                    valid_time=valid_value,
                    lead_time_hours=lead_value,
                    indexers=selector,
                )
            )
        return records, axes, set(ds.data_vars)


def build_catalog(label: str, source: Path) -> ForecastCatalog:
    """Build a validated forecast catalog, retaining warnings for unusable files."""
    files = discover_files(source)
    records: list[ForecastRecord] = []
    axes: dict[Path, DatasetAxes] = {}
    variables: dict[Path, set[str]] = {}
    failures: list[str] = []

    for path in tqdm(files, desc=f"Inspect {label}", unit="file"):
        try:
            file_records, file_axes, file_variables = forecast_records_from_file(path)
        except Exception as exc:
            failures.append(f"{path.name}: {exc}")
            continue
        records.extend(file_records)
        axes[path] = file_axes
        variables[path] = file_variables

    if not records:
        detail = "\n  - ".join(failures[:10])
        raise EvaluationError(
            f"No usable forecast records were found for {label} under {source}."
            + (f"\n  - {detail}" if detail else "")
        )
    for message in failures:
        DISCOVERY_WARNINGS.add(f"{label} skipped an unusable NetCDF file: {message}")

    tolerance = float(EVAL["lead_time_tolerance_hours"])
    seen: dict[tuple[int, int], ForecastRecord] = {}
    for record in records:
        init_key = int(record.initialization_time.astype("datetime64[ns]").astype(np.int64))
        lead_key = int(round(record.lead_time_hours / max(tolerance, 1.0e-12)))
        key = (init_key, lead_key)
        if key in seen:
            other = seen[key]
            raise EvaluationError(
                f"{label} has duplicate forecasts for init={record.initialization_time}, "
                f"lead={record.lead_time_hours:g} h in {other.path} and {record.path}. "
                "Narrow evaluation.file_pattern or remove duplicate products."
            )
        seen[key] = record

    return ForecastCatalog(label, files, records, axes, variables)


DISCOVERY_WARNINGS = WarningLog()


In [5]:
def discover_truth_time(ds: xr.Dataset, path: Path) -> str:
    """Identify the one-dimensional ground-truth valid-time coordinate."""
    data_cfg = RAW_CONFIG.get("data", {}) or {}
    configured = str(data_cfg.get("time_dim", "")).strip()
    names = tuple(OrderedDict.fromkeys([*VALID_NAMES, configured, *TIME_NAMES]))
    name = first_present(
        ds, [value for value in names if value], require_datetime=True
    )
    if name is None:
        raise EvaluationError(
            f"{path}: no datetime ground-truth time coordinate found. "
            f"Coordinates={list(ds.coords)}."
        )
    if ds[name].ndim != 1:
        raise EvaluationError(
            f"{path}: ground-truth valid-time coordinate `{name}` must be one-dimensional; "
            f"found dims={ds[name].dims}."
        )
    return name


def truth_time_lookup(ds: xr.Dataset, time_name: str, path: Path) -> dict[int, int]:
    """Map exact ground-truth valid times to positional indices."""
    values = np.asarray(ds[time_name].values).astype("datetime64[ns]")
    lookup: dict[int, int] = {}
    for index, value in enumerate(values):
        if np.isnat(value):
            raise EvaluationError(f"{path}: ground-truth `{time_name}` contains NaT.")
        key = int(value.astype(np.int64))
        if key in lookup:
            raise EvaluationError(
                f"{path}: duplicate ground-truth valid time {value}; matching is ambiguous."
            )
        lookup[key] = index
    return lookup


def summarize_dataset(
    label: str,
    files: int,
    variables: Iterable[str],
    dimensions: Mapping[str, int],
    records: Sequence[ForecastRecord] | None = None,
) -> None:
    """Print a concise discovered-dataset summary."""
    variables_text = ", ".join(sorted(variables))
    print(f"\n{label}")
    print(f"  files: {files}")
    print(f"  dimensions (representative): {dict(dimensions)}")
    print(f"  variables: {variables_text}")
    if records:
        inits = sorted({record.initialization_time for record in records})
        leads = sorted({round(record.lead_time_hours, 9) for record in records})
        print(f"  initialization times: {len(inits)} ({inits[0]} to {inits[-1]})")
        print(f"  discovered lead hours: {leads}")


GROUND_TRUTH_PATH = Path(EVAL["ground_truth_path"])
if not GROUND_TRUTH_PATH.is_file():
    raise FileNotFoundError(
        f"Ground-truth NetCDF does not exist: {GROUND_TRUTH_PATH}. "
        "Set evaluation.ground_truth_path or create data/{case_name}/test.nc."
    )

# Inputs stay read-only. xarray opens variables lazily; individual fields are
# loaded only when a matched forecast case is evaluated.
TRUTH_DS = xr.open_dataset(GROUND_TRUTH_PATH, decode_times=True, decode_timedelta=True)
TRUTH_AXES = discover_axes(TRUTH_DS, GROUND_TRUTH_PATH)
TRUTH_TIME = discover_truth_time(TRUTH_DS, GROUND_TRUTH_PATH)
TRUTH_LOOKUP = truth_time_lookup(TRUTH_DS, TRUTH_TIME, GROUND_TRUTH_PATH)

BASELINE = build_catalog("baseline", Path(EVAL["baseline_rollout_path"]))
FINETUNED = build_catalog("fine-tuned", Path(EVAL["finetuned_rollout_path"]))

baseline_example = BASELINE.files[0]
finetuned_example = FINETUNED.files[0]
with xr.open_dataset(baseline_example) as example:
    summarize_dataset(
        "Ground truth",
        1,
        TRUTH_DS.data_vars,
        TRUTH_DS.sizes,
    )
    summarize_dataset(
        "Baseline rollouts",
        len(BASELINE.files),
        set().union(*BASELINE.variables.values()),
        example.sizes,
        BASELINE.records,
    )
with xr.open_dataset(finetuned_example) as example:
    summarize_dataset(
        "Fine-tuned rollouts",
        len(FINETUNED.files),
        set().union(*FINETUNED.variables.values()),
        example.sizes,
        FINETUNED.records,
    )


Inspect baseline:   0%|          | 0/854 [00:00<?, ?file/s]

Inspect baseline:   1%|          | 10/854 [00:00<00:08, 96.57file/s]

Inspect baseline:   2%|▏         | 20/854 [00:00<00:08, 97.72file/s]

Inspect baseline:   4%|▎         | 30/854 [00:00<00:08, 98.25file/s]

Inspect baseline:   5%|▍         | 40/854 [00:00<00:08, 98.68file/s]

...

Inspect fine-tuned:  78%|███████▊  | 144/185 [00:00<00:00, 153.59file/s]

Inspect fine-tuned:  86%|████████▋ | 160/185 [00:01<00:00, 153.60file/s]

Inspect fine-tuned:  95%|█████████▌| 176/185 [00:01<00:00, 153.86file/s]

Inspect fine-tuned: 100%|██████████| 185/185 [00:01<00:00, 154.01file/s]


Ground truth
  files: 1
  dimensions (representative): {'time': 185, 'latitude': 53, 'longitude': 70, 'level': 13}
  variables: msl, no2, q, t, t2m, tcno2, u, u10, v, v10, z

Baseline rollouts
  files: 854
  dimensions (representative): {'time': 6, 'latitude': 450, 'longitude': 900, 'level': 13}
  variables: 10u, 10v, 2t, co, go3, gtco3, msl, no, no2, pm1, pm10, pm2p5, q, so2, t, tc_no, tcco, tcno2, tcso2, u, v, z
  initialization times: 854 (2023-08-01T00:00:00.000000000 to 2024-09-30T12:00:00.000000000)
  discovered lead hours: [12.0, 24.0, 36.0, 48.0, 60.0, 72.0]

Fine-tuned rollouts
  files: 185
  dimensions (representative): {'time': 6, 'latitude': 51, 'longitude': 69, 'level': 13}
  variables: no2, tcno2
  initialization times: 184 (2024-07-01T00:00:00.000000000 to 2024-09-30T12:00:00.000000000)
  discovered lead hours: [12.0, 24.0, 36.0, 48.0, 60.0, 72.0]


## 3. Targets, levels, and coordinate validation

Explicit YAML target definitions take precedence:

1. `evaluation.target_variables`
2. `data.target_variables`
3. compatible variables shared by all three products

For atmospheric targets, level selection uses
`evaluation.levels`, then target `loss_levels`, then
`data.atmos_levels`, then the coordinate intersection. Surface targets are
labeled `surface`.


In [6]:
@dataclass(frozen=True)
class VariableBinding:
    """Resolved name of one logical target in each product."""

    spec: VariableSpec
    truth_name: str
    baseline_name: str
    finetuned_name: str
    units: str


def parse_variable_specs(items: Any) -> list[VariableSpec]:
    """Parse string or structured YAML target-variable entries."""
    if items is None:
        return []
    if isinstance(items, (str, Mapping)):
        items = [items]
    specs: list[VariableSpec] = []
    for item in items:
        if isinstance(item, str):
            name = item.strip()
            aliases = (name,)
            kind = None
            loss_levels = None
        elif isinstance(item, Mapping):
            name = str(item.get("dataset_name") or item.get("name") or "").strip()
            if not name:
                raise ValueError(f"Invalid target-variable definition: {item!r}")
            aliases = tuple(
                OrderedDict.fromkeys(
                    str(value).strip()
                    for value in (
                        name,
                        item.get("aurora_name"),
                        item.get("variable"),
                        item.get("name"),
                    )
                    if value is not None and str(value).strip()
                )
            )
            kind_value = item.get("kind")
            kind = None if kind_value is None else str(kind_value).strip().lower()
            raw_levels = item.get("loss_levels")
            loss_levels = (
                None
                if raw_levels is None
                else tuple(float(value) for value in raw_levels)
            )
        else:
            raise TypeError(f"Unsupported target-variable definition: {item!r}")
        specs.append(VariableSpec(name, aliases, kind, loss_levels))
    return specs


def variable_has_spatial_dims(ds: xr.Dataset, name: str, axes: DatasetAxes) -> bool:
    """Return whether a variable contains the discovered horizontal dimensions."""
    dims = set(ds[name].dims)
    return axes.latitude in dims and axes.longitude in dims


def automatic_variable_specs() -> list[VariableSpec]:
    """Find compatible, identically named variables shared by all products."""
    baseline_union = set().union(*BASELINE.variables.values())
    finetuned_union = set().union(*FINETUNED.variables.values())
    names = sorted(set(TRUTH_DS.data_vars) & baseline_union & finetuned_union)
    compatible = [
        name
        for name in names
        if variable_has_spatial_dims(TRUTH_DS, name, TRUTH_AXES)
    ]
    if not compatible:
        raise EvaluationError(
            "No compatible variables are shared by truth, baseline, and fine-tuned "
            "datasets. Set evaluation.target_variables with aliases if names differ."
        )
    return [VariableSpec(name, (name,)) for name in compatible]


def first_alias_present(aliases: Sequence[str], available: set[str]) -> str | None:
    """Return the first configured alias in an available-name set."""
    return next((alias for alias in aliases if alias in available), None)


def resolve_bindings(specs: Sequence[VariableSpec]) -> list[VariableBinding]:
    """Resolve every explicit target across truth, baseline, and fine-tuned products."""
    baseline_union = set().union(*BASELINE.variables.values())
    finetuned_union = set().union(*FINETUNED.variables.values())
    bindings: list[VariableBinding] = []
    failures: list[str] = []
    for spec in specs:
        truth_name = first_alias_present(spec.aliases, set(TRUTH_DS.data_vars))
        baseline_name = first_alias_present(spec.aliases, baseline_union)
        finetuned_name = first_alias_present(spec.aliases, finetuned_union)
        missing = [
            label
            for label, value in (
                ("ground truth", truth_name),
                ("baseline", baseline_name),
                ("fine-tuned", finetuned_name),
            )
            if value is None
        ]
        if missing:
            failures.append(
                f"`{spec.dataset_name}` (aliases={list(spec.aliases)}) missing from "
                + ", ".join(missing)
            )
            continue
        assert truth_name and baseline_name and finetuned_name
        if not variable_has_spatial_dims(TRUTH_DS, truth_name, TRUTH_AXES):
            failures.append(
                f"`{truth_name}` in ground truth lacks both latitude and longitude dimensions"
            )
            continue
        attrs = TRUTH_DS[truth_name].attrs
        units = str(attrs.get("units") or attrs.get("GRIB_units") or "").strip()
        bindings.append(
            VariableBinding(spec, truth_name, baseline_name, finetuned_name, units)
        )
    if failures:
        raise EvaluationError(
            "Configured target variables could not be matched:\n  - "
            + "\n  - ".join(failures)
            + "\nCheck dataset variable names or add dataset_name/aurora_name aliases."
        )
    return bindings


configured_targets = EVAL.get("target_variables")
if configured_targets is None:
    configured_targets = (RAW_CONFIG.get("data", {}) or {}).get("target_variables")
TARGET_SPECS = (
    parse_variable_specs(configured_targets)
    if configured_targets is not None
    else automatic_variable_specs()
)
BINDINGS = resolve_bindings(TARGET_SPECS)


def variable_level_name(ds: xr.Dataset, variable: str, axes: DatasetAxes) -> str | None:
    """Return the vertical dimension used by a variable, if any."""
    if axes.level and axes.level in ds[variable].dims:
        return axes.level
    for candidate in LEVEL_NAMES:
        if candidate in ds[variable].dims and candidate in ds:
            return candidate
    return None


def explicit_levels_for(binding: VariableBinding) -> list[float] | None:
    """Resolve YAML-requested levels for one target, preserving precedence."""
    configured = EVAL.get("levels")
    if isinstance(configured, Mapping):
        value = next(
            (
                configured[key]
                for key in (binding.spec.dataset_name, *binding.spec.aliases)
                if key in configured
            ),
            None,
        )
        if value is not None:
            return [float(level) for level in value]
    elif configured is not None:
        return [float(level) for level in configured]

    if binding.spec.loss_levels is not None:
        return list(binding.spec.loss_levels)
    atmos_levels = (RAW_CONFIG.get("data", {}) or {}).get("atmos_levels")
    if atmos_levels is not None:
        return [float(level) for level in atmos_levels]
    return None


def representative_level_values(
    catalog: ForecastCatalog, variable: str
) -> list[np.ndarray]:
    """Read vertical coordinates from files that contain a target variable."""
    values: list[np.ndarray] = []
    for path, names in catalog.variables.items():
        if variable not in names:
            continue
        with xr.open_dataset(path, decode_times=True, decode_timedelta=True) as ds:
            level_name = variable_level_name(ds, variable, catalog.axes[path])
            if level_name:
                values.append(np.asarray(ds[level_name].values, dtype=float).reshape(-1))
        if values:
            break
    return values


def shared_levels(binding: VariableBinding) -> list[float | None]:
    """Return requested/shared pressure levels, or [None] for a surface variable."""
    truth_level = variable_level_name(TRUTH_DS, binding.truth_name, TRUTH_AXES)
    surface_kind = binding.spec.kind in {"surf", "surface"}
    if truth_level is None or surface_kind:
        return [None]

    requested = explicit_levels_for(binding)
    if requested is not None:
        return requested

    arrays = [np.asarray(TRUTH_DS[truth_level].values, dtype=float).reshape(-1)]
    arrays += representative_level_values(BASELINE, binding.baseline_name)
    arrays += representative_level_values(FINETUNED, binding.finetuned_name)
    tolerance = float(EVAL["level_tolerance"])
    result = []
    for value in arrays[0]:
        if all(np.any(np.isclose(value, other, atol=tolerance, rtol=0)) for other in arrays[1:]):
            result.append(float(value))
    if not result:
        raise EvaluationError(
            f"No shared vertical levels were found for `{binding.spec.dataset_name}`."
        )
    return result


LEVELS_BY_VARIABLE = {
    binding.spec.dataset_name: shared_levels(binding) for binding in BINDINGS
}

def discovered_level_units(binding: VariableBinding) -> str:
    """Return vertical-coordinate units, using hPa only for pressure-like axes."""
    level_name = variable_level_name(TRUTH_DS, binding.truth_name, TRUTH_AXES)
    if level_name is None:
        return ""
    units = str(TRUTH_DS[level_name].attrs.get("units") or "").strip()
    if units:
        return units
    pressure_like = (
        level_name in {"level", "pressure_level", "pressure", "isobaricInhPa", "plev"}
        or binding.spec.kind in {"atmos", "atmospheric"}
    )
    return "hPa" if pressure_like else ""


LEVEL_UNITS_BY_VARIABLE = {
    binding.spec.dataset_name: discovered_level_units(binding) for binding in BINDINGS
}

print("\nMatched targets")
for binding in BINDINGS:
    levels = [
        "surface" if value is None else f"{value:g}"
        for value in LEVELS_BY_VARIABLE[binding.spec.dataset_name]
    ]
    print(
        f"  {binding.spec.dataset_name}: truth={binding.truth_name}, "
        f"baseline={binding.baseline_name}, fine-tuned={binding.finetuned_name}, "
        f"levels={levels}, level_units="
        f"{LEVEL_UNITS_BY_VARIABLE[binding.spec.dataset_name] or '(not provided)'}, "
        f"variable_units={binding.units or '(not provided)'}"
    )



Matched targets
  no2: truth=no2, baseline=no2, fine-tuned=no2, levels=['1000', '925', '850'], level_units=hPa, variable_units=kg kg**-1
  tcno2: truth=tcno2, baseline=tcno2, fine-tuned=tcno2, levels=['surface'], level_units=(not provided), variable_units=kg m**-2


In [7]:
class DatasetCache:
    """Small LRU cache of lazily opened xarray datasets."""

    def __init__(self, max_open: int = 8):
        self.max_open = max(1, int(max_open))
        self._datasets: OrderedDict[Path, xr.Dataset] = OrderedDict()

    def get(self, path: Path) -> xr.Dataset:
        if path in self._datasets:
            self._datasets.move_to_end(path)
            return self._datasets[path]
        ds = xr.open_dataset(path, decode_times=True, decode_timedelta=True)
        self._datasets[path] = ds
        while len(self._datasets) > self.max_open:
            _, old = self._datasets.popitem(last=False)
            old.close()
        return ds

    def close(self) -> None:
        for ds in self._datasets.values():
            ds.close()
        self._datasets.clear()

    def __enter__(self) -> "DatasetCache":
        return self

    def __exit__(self, *_: Any) -> None:
        self.close()


def select_level(
    da: xr.DataArray,
    level_name: str | None,
    requested_level: float | None,
    *,
    context: str,
) -> xr.DataArray:
    """Select one exact-within-tolerance vertical level without interpolation."""
    if requested_level is None:
        if level_name and level_name in da.dims:
            if da.sizes[level_name] == 1:
                return da.isel({level_name: 0}, drop=True)
            raise EvaluationError(
                f"{context}: variable has {da.sizes[level_name]} vertical levels but is "
                "configured as a surface variable."
            )
        return da
    if level_name is None or level_name not in da.dims:
        raise EvaluationError(
            f"{context}: requested level {requested_level:g}, but variable has no "
            "recognized vertical dimension."
        )
    values = np.asarray(da[level_name].values, dtype=float).reshape(-1)
    matches = np.flatnonzero(
        np.isclose(values, requested_level, atol=float(EVAL["level_tolerance"]), rtol=0)
    )
    if len(matches) != 1:
        raise EvaluationError(
            f"{context}: requested level {requested_level:g} matched {len(matches)} "
            f"coordinates in {values.tolist()}. Adjust evaluation.levels or "
            "evaluation.level_tolerance."
        )
    return da.isel({level_name: int(matches[0])}, drop=True)


def reduce_extra_dimensions(
    da: xr.DataArray,
    axes: DatasetAxes,
    *,
    context: str,
) -> xr.DataArray:
    """Reduce ensemble/configured dimensions and squeeze only singleton extras."""
    member = axes.member if axes.member in da.dims else None
    if member:
        mode = EVAL["ensemble_reduction"]
        if mode == "mean":
            da = da.mean(member, skipna=True, keep_attrs=True)
        elif mode == "median":
            da = da.median(member, skipna=True, keep_attrs=True)
        else:
            da = da.isel({member: 0}, drop=True)

    allowed = {axes.latitude, axes.longitude}
    extras = [dim for dim in da.dims if dim not in allowed]
    for dim in extras:
        if dim in EVAL["extra_dim_indexers"]:
            index = int(EVAL["extra_dim_indexers"][dim])
            if not 0 <= index < da.sizes[dim]:
                raise EvaluationError(
                    f"{context}: extra_dim_indexers[{dim!r}]={index} is outside "
                    f"dimension size {da.sizes[dim]}."
                )
            da = da.isel({dim: index}, drop=True)
        elif da.sizes[dim] == 1:
            da = da.isel({dim: 0}, drop=True)
        else:
            raise EvaluationError(
                f"{context}: unresolved non-spatial dimension `{dim}` has size "
                f"{da.sizes[dim]}. Add evaluation.extra_dim_indexers or configure "
                "the ensemble dimension."
            )
    return da


def canonicalize_horizontal(
    da: xr.DataArray, axes: DatasetAxes, *, context: str
) -> xr.DataArray:
    """Rename horizontal axes, normalize longitudes, sort, and validate uniqueness."""
    rename = {}
    if axes.latitude != "latitude":
        rename[axes.latitude] = "latitude"
    if axes.longitude != "longitude":
        rename[axes.longitude] = "longitude"
    da = da.rename(rename)

    latitude = np.asarray(da["latitude"].values, dtype=float)
    longitude = np.asarray(da["longitude"].values, dtype=float)
    if not np.all(np.isfinite(latitude)) or not np.all(np.isfinite(longitude)):
        raise EvaluationError(f"{context}: latitude/longitude contains non-finite values.")
    normalized_lon = ((longitude + 180.0) % 360.0) - 180.0
    da = da.assign_coords(longitude=normalized_lon)
    da = da.sortby("latitude").sortby("longitude")

    tolerance = float(EVAL["coordinate_tolerance"])
    for name in ("latitude", "longitude"):
        values = np.asarray(da[name].values, dtype=float)
        if len(values) > 1 and np.any(np.diff(values) <= tolerance):
            raise EvaluationError(
                f"{context}: normalized `{name}` coordinates are duplicated or closer "
                f"than coordinate_tolerance={tolerance:g}."
            )
    return da.transpose("latitude", "longitude")


def extract_truth_field(
    variable: str,
    time_index: int,
    level: float | None,
    *,
    context: str,
) -> xr.DataArray:
    """Load one two-dimensional ground-truth field."""
    da = TRUTH_DS[variable]
    if TRUTH_TIME in da.dims:
        da = da.isel({TRUTH_TIME: time_index}, drop=True)
    else:
        raise EvaluationError(f"{context}: truth variable lacks time dimension `{TRUTH_TIME}`.")
    level_name = variable_level_name(TRUTH_DS, variable, TRUTH_AXES)
    da = select_level(da, level_name, level, context=context)
    da = reduce_extra_dimensions(da, TRUTH_AXES, context=context)
    return canonicalize_horizontal(da, TRUTH_AXES, context=context)


def extract_forecast_field(
    catalog: ForecastCatalog,
    record: ForecastRecord,
    variable: str,
    level: float | None,
    cache: DatasetCache,
    *,
    context: str,
) -> xr.DataArray:
    """Load one two-dimensional forecast field for a catalog record."""
    ds = cache.get(record.path)
    if variable not in ds:
        raise EvaluationError(f"{context}: `{variable}` is missing from {record.path.name}.")
    da = ds[variable]
    for dim, index in record.selector.items():
        if dim in da.dims:
            da = da.isel({dim: index}, drop=True)
    axes = catalog.axes[record.path]
    level_name = variable_level_name(ds, variable, axes)
    da = select_level(da, level_name, level, context=context)
    da = reduce_extra_dimensions(da, axes, context=context)
    return canonicalize_horizontal(da, axes, context=context)


def matching_indices(
    reference: np.ndarray, candidate: np.ndarray, tolerance: float
) -> tuple[np.ndarray, np.ndarray]:
    """Return unique coordinate matches from reference to candidate."""
    ref_indices: list[int] = []
    candidate_indices: list[int] = []
    used: set[int] = set()
    for ref_index, value in enumerate(reference):
        distances = np.abs(candidate - value)
        possible = np.flatnonzero(distances <= tolerance)
        if len(possible) == 1 and int(possible[0]) not in used:
            ref_indices.append(ref_index)
            candidate_indices.append(int(possible[0]))
            used.add(int(possible[0]))
        elif len(possible) > 1:
            raise EvaluationError(
                f"Coordinate {value:g} has multiple matches within tolerance {tolerance:g}."
            )
    return np.asarray(ref_indices, dtype=int), np.asarray(candidate_indices, dtype=int)


def align_fields(
    truth: xr.DataArray,
    baseline: xr.DataArray,
    finetuned: xr.DataArray,
    *,
    context: str,
) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    """Validate and align three horizontal fields without interpolation."""
    tolerance = float(EVAL["coordinate_tolerance"])
    arrays = [truth, baseline, finetuned]
    exact = True
    for dim in ("latitude", "longitude"):
        reference = np.asarray(truth[dim].values, dtype=float)
        for candidate in arrays[1:]:
            values = np.asarray(candidate[dim].values, dtype=float)
            exact &= len(reference) == len(values) and np.allclose(
                reference, values, atol=tolerance, rtol=0
            )

    if exact:
        baseline = baseline.assign_coords(
            latitude=truth.latitude, longitude=truth.longitude
        )
        finetuned = finetuned.assign_coords(
            latitude=truth.latitude, longitude=truth.longitude
        )
        return truth, baseline, finetuned

    if EVAL["spatial_alignment"] == "exact":
        shapes = [tuple(array.sizes[dim] for dim in ("latitude", "longitude")) for array in arrays]
        raise EvaluationError(
            f"{context}: horizontal grids do not match (truth/baseline/fine-tuned "
            f"shapes={shapes}). No resizing or interpolation was attempted. If the "
            "products intentionally crop boundary coordinates, explicitly set "
            "evaluation.spatial_alignment: coordinate_intersection."
        )

    # Explicit coordinate intersection: match by values, never by array position.
    truth_lat = np.asarray(truth.latitude.values, dtype=float)
    truth_lon = np.asarray(truth.longitude.values, dtype=float)
    base_lat = np.asarray(baseline.latitude.values, dtype=float)
    base_lon = np.asarray(baseline.longitude.values, dtype=float)
    fine_lat = np.asarray(finetuned.latitude.values, dtype=float)
    fine_lon = np.asarray(finetuned.longitude.values, dtype=float)

    truth_lat_base, base_lat_idx = matching_indices(truth_lat, base_lat, tolerance)
    truth_lat_fine, fine_lat_idx = matching_indices(truth_lat, fine_lat, tolerance)
    truth_lon_base, base_lon_idx = matching_indices(truth_lon, base_lon, tolerance)
    truth_lon_fine, fine_lon_idx = matching_indices(truth_lon, fine_lon, tolerance)

    shared_lat_truth = np.intersect1d(truth_lat_base, truth_lat_fine)
    shared_lon_truth = np.intersect1d(truth_lon_base, truth_lon_fine)
    if len(shared_lat_truth) == 0 or len(shared_lon_truth) == 0:
        raise EvaluationError(f"{context}: no common latitude/longitude grid intersection.")

    base_lat_map = dict(zip(truth_lat_base.tolist(), base_lat_idx.tolist()))
    fine_lat_map = dict(zip(truth_lat_fine.tolist(), fine_lat_idx.tolist()))
    base_lon_map = dict(zip(truth_lon_base.tolist(), base_lon_idx.tolist()))
    fine_lon_map = dict(zip(truth_lon_fine.tolist(), fine_lon_idx.tolist()))

    truth = truth.isel(latitude=shared_lat_truth, longitude=shared_lon_truth)
    baseline = baseline.isel(
        latitude=[base_lat_map[int(i)] for i in shared_lat_truth],
        longitude=[base_lon_map[int(i)] for i in shared_lon_truth],
    )
    finetuned = finetuned.isel(
        latitude=[fine_lat_map[int(i)] for i in shared_lat_truth],
        longitude=[fine_lon_map[int(i)] for i in shared_lon_truth],
    )
    baseline = baseline.assign_coords(latitude=truth.latitude, longitude=truth.longitude)
    finetuned = finetuned.assign_coords(latitude=truth.latitude, longitude=truth.longitude)
    METRIC_WARNINGS.add(
        f"Explicit coordinate_intersection selected a common "
        f"{truth.sizes['latitude']}x{truth.sizes['longitude']} coordinate grid; "
        "no interpolation was used."
    )
    return truth, baseline, finetuned


## 4. Metric definitions and evaluation

For every variable/level/lead combination, both products use the same
forecast cases and the same finite-point mask. Bias, MAE, and RMSE aggregate
all valid points across all matched cases. Spatial correlation is Pearson
correlation across the horizontal grid for each forecast case, followed by
an equal-weight mean across cases with defined correlation.

Improvement signs are defined so **positive is better**:

- bias improvement = `abs(baseline bias) - abs(fine-tuned bias)`
- MAE/RMSE improvement = `baseline - fine-tuned`
- spatial-correlation improvement = `fine-tuned - baseline`
- percentage reduction = `100 * (baseline - fine-tuned) / abs(baseline)`

A percentage is `NaN` when the baseline denominator is less than or equal
to `percentage_epsilon`. Raw signed changes (`fine-tuned - baseline`) are
retained in additional columns.


In [8]:
def record_lookup(catalog: ForecastCatalog, requested_lead: float) -> dict[int, ForecastRecord]:
    """Index catalog records by initialization time for one lead."""
    tolerance = float(EVAL["lead_time_tolerance_hours"])
    selected = [
        record
        for record in catalog.records
        if math.isclose(
            record.lead_time_hours, requested_lead, abs_tol=tolerance, rel_tol=0
        )
    ]
    return {
        int(record.initialization_time.astype("datetime64[ns]").astype(np.int64)): record
        for record in selected
    }


def expected_valid_time(initialization_ns: int, lead: float) -> np.datetime64:
    """Compute valid time from initialization plus lead hours."""
    init = np.datetime64(initialization_ns, "ns")
    delta_ns = int(round(lead * 3_600_000_000_000))
    return init + np.timedelta64(delta_ns, "ns")


def validate_record_time(
    record: ForecastRecord, expected: np.datetime64, *, context: str
) -> None:
    """Require a catalog valid time to equal init + requested lead."""
    difference_seconds = abs(
        float((record.valid_time - expected) / np.timedelta64(1, "s"))
    )
    if difference_seconds > float(EVAL["time_tolerance_seconds"]):
        raise EvaluationError(
            f"{context}: stored valid_time={record.valid_time} differs from "
            f"init+lead={expected} by {difference_seconds:g} seconds."
        )


def truth_index_for_time(valid_time: np.datetime64) -> int | None:
    """Find an exact-within-tolerance truth index for a forecast valid time."""
    key = int(valid_time.astype("datetime64[ns]").astype(np.int64))
    if key in TRUTH_LOOKUP:
        return TRUTH_LOOKUP[key]
    tolerance_ns = int(float(EVAL["time_tolerance_seconds"]) * 1_000_000_000)
    candidates = [
        (abs(stored - key), index) for stored, index in TRUTH_LOOKUP.items()
        if abs(stored - key) <= tolerance_ns
    ]
    if len(candidates) == 1:
        return candidates[0][1]
    if len(candidates) > 1:
        raise EvaluationError(
            f"Ground truth has multiple valid times within tolerance of {valid_time}."
        )
    return None


def spatial_correlation(observation: np.ndarray, forecast: np.ndarray) -> float:
    """Pearson correlation across finite paired spatial points."""
    mask = np.isfinite(observation) & np.isfinite(forecast)
    x = observation[mask].astype(float)
    y = forecast[mask].astype(float)
    if x.size < 2 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def safe_percent_reduction(baseline: float, finetuned: float) -> float:
    """Return percentage reduction, or NaN for a near-zero baseline."""
    epsilon = float(EVAL["percentage_epsilon"])
    if not np.isfinite(baseline) or not np.isfinite(finetuned) or abs(baseline) <= epsilon:
        return np.nan
    return 100.0 * (baseline - finetuned) / abs(baseline)


def level_label(level: float | None) -> str:
    """Format a tidy, unambiguous level label."""
    return "surface" if level is None else f"{float(level):g}"


def empty_result_row(binding: VariableBinding, level: float | None, lead: float) -> dict[str, Any]:
    """Create a coverage row for a skipped/missing combination."""
    row: dict[str, Any] = {
        "case_name": CASE_NAME,
        "variable": binding.spec.dataset_name,
        "level": level_label(level),
        "level_value": np.nan if level is None else float(level),
        "level_units": (
            "" if level is None
            else LEVEL_UNITS_BY_VARIABLE[binding.spec.dataset_name]
        ),
        "variable_units": binding.units,
        "lead_time_hours": float(lead),
        "number_of_forecasts": 0,
        "number_of_valid_points": 0,
        "number_of_correlation_forecasts": 0,
    }
    for name in (
        "baseline_bias", "finetuned_bias", "bias_change", "bias_improvement",
        "bias_improvement_percent", "baseline_mae", "finetuned_mae",
        "mae_change", "mae_improvement", "mae_improvement_percent",
        "baseline_rmse", "finetuned_rmse", "rmse_change",
        "rmse_improvement", "rmse_improvement_percent",
        "baseline_spatial_correlation", "finetuned_spatial_correlation",
        "spatial_correlation_change", "spatial_correlation_improvement",
    ):
        row[name] = np.nan
    return row


def per_case_metrics(
    truth: np.ndarray, baseline: np.ndarray, finetuned: np.ndarray
) -> dict[str, float | int]:
    """Compute paired metrics for one forecast field."""
    mask = np.isfinite(truth) & np.isfinite(baseline) & np.isfinite(finetuned)
    count = int(mask.sum())
    if count == 0:
        return {"number_of_valid_points": 0}
    obs = truth[mask].astype(float)
    base = baseline[mask].astype(float)
    fine = finetuned[mask].astype(float)
    base_error = base - obs
    fine_error = fine - obs
    return {
        "number_of_valid_points": count,
        "baseline_bias": float(base_error.mean()),
        "finetuned_bias": float(fine_error.mean()),
        "baseline_mae": float(np.abs(base_error).mean()),
        "finetuned_mae": float(np.abs(fine_error).mean()),
        "baseline_rmse": float(np.sqrt(np.square(base_error).mean())),
        "finetuned_rmse": float(np.sqrt(np.square(fine_error).mean())),
        "baseline_spatial_correlation": spatial_correlation(obs, base),
        "finetuned_spatial_correlation": spatial_correlation(obs, fine),
    }


METRIC_WARNINGS = WarningLog()
SKIPPED: list[dict[str, Any]] = []


def evaluate_combination(
    binding: VariableBinding,
    level: float | None,
    lead: float,
    baseline_cache: DatasetCache,
    finetuned_cache: DatasetCache,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """Evaluate a single variable/level/lead across all common initializations."""
    base_records = record_lookup(BASELINE, lead)
    fine_records = record_lookup(FINETUNED, lead)
    common_inits = sorted(set(base_records) & set(fine_records))
    missing_base = len(set(fine_records) - set(base_records))
    missing_fine = len(set(base_records) - set(fine_records))
    if missing_base or missing_fine:
        METRIC_WARNINGS.add(
            f"{binding.spec.dataset_name}/{level_label(level)}/{lead:g}h has "
            f"{missing_base} initialization(s) missing baseline and "
            f"{missing_fine} missing fine-tuned forecasts; only common cases were used."
        )

    if not common_inits:
        reason = (
            f"No common baseline/fine-tuned initialization at lead {lead:g} h for "
            f"{binding.spec.dataset_name}/{level_label(level)}."
        )
        METRIC_WARNINGS.add(reason)
        SKIPPED.append(
            {"variable": binding.spec.dataset_name, "level": level_label(level),
             "lead_time_hours": lead, "reason": reason}
        )
        return empty_result_row(binding, level, lead), []

    sum_base_error = sum_fine_error = 0.0
    sum_base_abs = sum_fine_abs = 0.0
    sum_base_sq = sum_fine_sq = 0.0
    valid_points = 0
    forecasts = 0
    base_correlations: list[float] = []
    fine_correlations: list[float] = []
    detail_rows: list[dict[str, Any]] = []
    missing_truth = 0

    for init_ns in common_inits:
        expected = expected_valid_time(init_ns, lead)
        base_record = base_records[init_ns]
        fine_record = fine_records[init_ns]
        context = (
            f"{binding.spec.dataset_name}/{level_label(level)}/"
            f"init={np.datetime64(init_ns, 'ns')}/lead={lead:g}h"
        )
        validate_record_time(base_record, expected, context=f"baseline {context}")
        validate_record_time(fine_record, expected, context=f"fine-tuned {context}")
        truth_index = truth_index_for_time(expected)
        if truth_index is None:
            missing_truth += 1
            continue
        truth = extract_truth_field(
            binding.truth_name, truth_index, level, context=f"truth {context}"
        )
        baseline = extract_forecast_field(
            BASELINE, base_record, binding.baseline_name, level,
            baseline_cache, context=f"baseline {context}"
        )
        finetuned = extract_forecast_field(
            FINETUNED, fine_record, binding.finetuned_name, level,
            finetuned_cache, context=f"fine-tuned {context}"
        )
        truth, baseline, finetuned = align_fields(
            truth, baseline, finetuned, context=context
        )

        truth_values = np.asarray(truth.values)
        base_values = np.asarray(baseline.values)
        fine_values = np.asarray(finetuned.values)
        mask = (
            np.isfinite(truth_values)
            & np.isfinite(base_values)
            & np.isfinite(fine_values)
        )
        count = int(mask.sum())
        if count == 0:
            METRIC_WARNINGS.add(
                f"{binding.spec.dataset_name}/{level_label(level)}/{lead:g}h "
                "had a matched forecast case with zero common finite points."
            )
            continue

        obs = truth_values[mask].astype(float)
        base = base_values[mask].astype(float)
        fine = fine_values[mask].astype(float)
        base_error = base - obs
        fine_error = fine - obs
        sum_base_error += float(base_error.sum())
        sum_fine_error += float(fine_error.sum())
        sum_base_abs += float(np.abs(base_error).sum())
        sum_fine_abs += float(np.abs(fine_error).sum())
        sum_base_sq += float(np.square(base_error).sum())
        sum_fine_sq += float(np.square(fine_error).sum())
        valid_points += count
        forecasts += 1

        base_corr = spatial_correlation(obs, base)
        fine_corr = spatial_correlation(obs, fine)
        if np.isfinite(base_corr) and np.isfinite(fine_corr):
            base_correlations.append(base_corr)
            fine_correlations.append(fine_corr)
        else:
            METRIC_WARNINGS.add(
                f"{binding.spec.dataset_name}/{level_label(level)}/{lead:g}h "
                "had a forecast with undefined spatial correlation (constant or "
                "insufficient finite field)."
            )

        if EVAL["save_per_case_metrics"]:
            detail = per_case_metrics(truth_values, base_values, fine_values)
            detail_rows.append(
                {
                    "case_name": CASE_NAME,
                    "variable": binding.spec.dataset_name,
                    "level": level_label(level),
                    "level_value": np.nan if level is None else float(level),
                    "lead_time_hours": float(lead),
                    "initialization_time": str(np.datetime64(init_ns, "ns")),
                    "valid_time": str(expected),
                    **detail,
                }
            )

    if missing_truth:
        METRIC_WARNINGS.add(
            f"{binding.spec.dataset_name}/{level_label(level)}/{lead:g}h skipped "
            f"{missing_truth} forecast(s) whose init+lead valid time was absent from truth."
        )
    if forecasts == 0 or valid_points == 0:
        reason = (
            f"No valid paired data remained for {binding.spec.dataset_name}/"
            f"{level_label(level)}/{lead:g}h."
        )
        SKIPPED.append(
            {"variable": binding.spec.dataset_name, "level": level_label(level),
             "lead_time_hours": lead, "reason": reason}
        )
        return empty_result_row(binding, level, lead), detail_rows

    baseline_bias = sum_base_error / valid_points
    finetuned_bias = sum_fine_error / valid_points
    baseline_mae = sum_base_abs / valid_points
    finetuned_mae = sum_fine_abs / valid_points
    baseline_rmse = math.sqrt(sum_base_sq / valid_points)
    finetuned_rmse = math.sqrt(sum_fine_sq / valid_points)
    baseline_corr = float(np.mean(base_correlations)) if base_correlations else np.nan
    finetuned_corr = float(np.mean(fine_correlations)) if fine_correlations else np.nan

    row = empty_result_row(binding, level, lead)
    row.update(
        {
            "number_of_forecasts": forecasts,
            "number_of_valid_points": valid_points,
            "number_of_correlation_forecasts": len(base_correlations),
            "baseline_bias": baseline_bias,
            "finetuned_bias": finetuned_bias,
            "bias_change": finetuned_bias - baseline_bias,
            "bias_improvement": abs(baseline_bias) - abs(finetuned_bias),
            "bias_improvement_percent": safe_percent_reduction(
                abs(baseline_bias), abs(finetuned_bias)
            ),
            "baseline_mae": baseline_mae,
            "finetuned_mae": finetuned_mae,
            "mae_change": finetuned_mae - baseline_mae,
            "mae_improvement": baseline_mae - finetuned_mae,
            "mae_improvement_percent": safe_percent_reduction(
                baseline_mae, finetuned_mae
            ),
            "baseline_rmse": baseline_rmse,
            "finetuned_rmse": finetuned_rmse,
            "rmse_change": finetuned_rmse - baseline_rmse,
            "rmse_improvement": baseline_rmse - finetuned_rmse,
            "rmse_improvement_percent": safe_percent_reduction(
                baseline_rmse, finetuned_rmse
            ),
            "baseline_spatial_correlation": baseline_corr,
            "finetuned_spatial_correlation": finetuned_corr,
            "spatial_correlation_change": finetuned_corr - baseline_corr,
            "spatial_correlation_improvement": finetuned_corr - baseline_corr,
        }
    )
    return row, detail_rows


In [9]:
TASKS = [
    (binding, level, lead)
    for binding in BINDINGS
    for level in LEVELS_BY_VARIABLE[binding.spec.dataset_name]
    for lead in EVAL["lead_times_hours"]
]

result_rows: list[dict[str, Any]] = []
per_case_rows: list[dict[str, Any]] = []
with DatasetCache(max_open=8) as baseline_cache, DatasetCache(max_open=8) as finetuned_cache:
    for binding, level, lead in tqdm(TASKS, desc="Evaluate targets", unit="selection"):
        row, details = evaluate_combination(
            binding, level, lead, baseline_cache, finetuned_cache
        )
        result_rows.append(row)
        per_case_rows.extend(details)

METRICS_DF = (
    pd.DataFrame(result_rows)
    .sort_values(["variable", "level_value", "level", "lead_time_hours"], na_position="first")
    .reset_index(drop=True)
)
PER_CASE_DF = pd.DataFrame(per_case_rows)

display_columns = [
    "variable", "level", "lead_time_hours", "number_of_forecasts",
    "number_of_valid_points", "baseline_rmse", "finetuned_rmse",
    "rmse_improvement_percent", "baseline_spatial_correlation",
    "finetuned_spatial_correlation", "spatial_correlation_improvement",
]
display(METRICS_DF[display_columns])


Evaluate targets:   0%|          | 0/24 [00:00<?, ?selection/s]

Evaluate targets:   4%|▍         | 1/24 [00:04<01:45,  4.60s/selection]

Evaluate targets:   8%|▊         | 2/24 [00:08<01:38,  4.48s/selection]

Evaluate targets:  12%|█▎        | 3/24 [00:13<01:33,  4.43s/selection]

Evaluate targets:  17%|█▋        | 4/24 [00:17<01:28,  4.40s/selection]

...

Evaluate targets:  92%|█████████▏| 22/24 [01:36<00:08,  4.44s/selection]

Evaluate targets:  96%|█████████▌| 23/24 [01:41<00:04,  4.41s/selection]

Evaluate targets: 100%|██████████| 24/24 [01:45<00:00,  4.38s/selection]

Evaluate targets: 100%|██████████| 24/24 [01:45<00:00,  4.40s/selection]

,variable,level,lead_time_hours,number_of_forecasts,number_of_valid_points,baseline_rmse,finetuned_rmse,rmse_improvement_percent,baseline_spatial_correlation,finetuned_spatial_correlation,spatial_correlation_improvement
0,no2,850,12.0,183,643977,8.267573e-10,8.324129e-10,-0.684068,0.803169,0.752362,-0.050807
1,no2,850,24.0,182,640458,9.106845e-10,8.352898e-10,8.278899,0.778102,0.750505,-0.027597
2,no2,850,36.0,181,636939,1.093619e-09,9.173395e-10,16.118892,0.694619,0.690143,-0.004476
3,no2,850,48.0,180,633420,1.152908e-09,9.198326e-10,20.216309,0.682542,0.686601,0.004059
4,no2,850,60.0,179,629901,1.279734e-09,9.501770e-10,25.751967,0.639235,0.664489,0.025254
5,no2,850,72.0,178,626382,1.315917e-09,9.516861e-10,27.678843,0.633493,0.662094,0.028601
6,no2,925,12.0,183,643977,1.229282e-09,1.271761e-09,-3.455564,0.810126,0.752878,-0.057248
7,no2,925,24.0,182,640458,1.338547e-09,1.276396e-09,4.643186,0.784739,0.749138,-0.035601
8,no2,925,36.0,181,636939,1.605167e-09,1.410707e-09,12.114663,0.703583,0.696911,-0.006672
9,no2,925,48.0,180,633420,1.676479e-09,1.414898e-09,15.602955,0.689040,0.693536,0.004497


## 5. Save tidy results and Markdown summary

The NetCDF uses a tidy `record` dimension rather than a sparse Cartesian
data cube, so surface and atmospheric targets can coexist without inventing
levels. String `level` values remain explicit (`surface`, `50`, `1000`,
etc.); `level_value` is numeric with `NaN` for surface variables.


In [10]:
def dataframe_to_netcdf(frame: pd.DataFrame, path: Path) -> None:
    """Write a heterogeneous tidy DataFrame to a NetCDF record dimension."""
    ds = xr.Dataset(coords={"record": np.arange(len(frame), dtype=np.int64)})
    for column in frame.columns:
        series = frame[column]
        if pd.api.types.is_numeric_dtype(series):
            values = series.to_numpy()
        else:
            values = series.fillna("").astype(str).to_numpy(dtype=str)
        ds[column] = xr.DataArray(values, dims=("record",))
    ds.attrs.update(
        {
            "title": "Baseline and fine-tuned CAMS rollout evaluation",
            "case_name": CASE_NAME,
            "metric_mask": "Common finite truth/baseline/fine-tuned points",
            "spatial_correlation": (
                "Mean of per-forecast Pearson correlations across spatial points"
            ),
            "improvement_sign": "Positive values indicate fine-tuned improvement",
            "percentage_reduction": (
                "100 * (baseline - finetuned) / abs(baseline); NaN when "
                "abs(baseline) <= percentage_epsilon"
            ),
            "spatial_alignment": str(EVAL["spatial_alignment"]),
            "coordinate_tolerance": float(EVAL["coordinate_tolerance"]),
            "percentage_epsilon": float(EVAL["percentage_epsilon"]),
        }
    )
    ds.to_netcdf(path)
    ds.close()


def markdown_table(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    """Render a small DataFrame as Markdown without requiring `tabulate`."""
    if frame.empty:
        return "_None._"
    shown = frame.loc[:, columns].copy()
    for column in shown:
        if pd.api.types.is_float_dtype(shown[column]):
            shown[column] = shown[column].map(
                lambda value: "" if pd.isna(value) else f"{value:.5g}"
            )
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = [
        "| " + " | ".join(str(value).replace("|", r"\|") for value in row) + " |"
        for row in shown.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])


def build_summary(frame: pd.DataFrame) -> str:
    """Build the required human-readable evaluation summary."""
    available = frame[frame["number_of_forecasts"] > 0].copy()
    by_target = (
        available.groupby(["variable", "level"], as_index=False)
        .agg(
            mean_rmse_improvement_percent=("rmse_improvement_percent", "mean"),
            mean_mae_improvement_percent=("mae_improvement_percent", "mean"),
            mean_bias_improvement=("bias_improvement", "mean"),
            mean_spatial_correlation_improvement=(
                "spatial_correlation_improvement", "mean"
            ),
            evaluated_leads=("lead_time_hours", "count"),
        )
    )
    largest = by_target.sort_values(
        "mean_rmse_improvement_percent", ascending=False, na_position="last"
    ).head(10)
    worse = by_target[
        (by_target["mean_rmse_improvement_percent"] < 0)
        | (by_target["mean_mae_improvement_percent"] < 0)
        | (by_target["mean_spatial_correlation_improvement"] < 0)
    ].sort_values("mean_rmse_improvement_percent", na_position="last")
    by_lead = (
        available.groupby("lead_time_hours", as_index=False)
        .agg(
            selections=("variable", "count"),
            mean_rmse_improvement_percent=("rmse_improvement_percent", "mean"),
            mean_mae_improvement_percent=("mae_improvement_percent", "mean"),
            mean_bias_improvement=("bias_improvement", "mean"),
            mean_spatial_correlation_improvement=(
                "spatial_correlation_improvement", "mean"
            ),
        )
        .sort_values("lead_time_hours")
    )
    missing = frame[frame["number_of_forecasts"] == 0][
        ["variable", "level", "lead_time_hours"]
    ]
    warning_messages = DISCOVERY_WARNINGS.messages() + METRIC_WARNINGS.messages()
    warning_text = (
        "\n".join(f"- {message}" for message in warning_messages)
        if warning_messages
        else "_No alignment or data-quality warnings._"
    )
    skipped_text = (
        "\n".join(
            f"- {item['variable']}/{item['level']}/{item['lead_time_hours']:g} h: "
            f"{item['reason']}"
            for item in SKIPPED
        )
        if SKIPPED
        else "_None._"
    )

    target_columns = [
        "variable", "level", "mean_rmse_improvement_percent",
        "mean_mae_improvement_percent", "mean_bias_improvement",
        "mean_spatial_correlation_improvement", "evaluated_leads",
    ]
    lead_columns = [
        "lead_time_hours", "selections", "mean_rmse_improvement_percent",
        "mean_mae_improvement_percent", "mean_bias_improvement",
        "mean_spatial_correlation_improvement",
    ]
    return f"""# CAMS rollout evaluation summary

- Case: `{CASE_NAME}`
- Ground truth: `{GROUND_TRUTH_PATH}`
- Baseline rollouts: `{EVAL['baseline_rollout_path']}`
- Fine-tuned rollouts: `{EVAL['finetuned_rollout_path']}`
- Spatial alignment policy: `{EVAL['spatial_alignment']}`
- Ensemble reduction: `{EVAL['ensemble_reduction']}`

## Metric interpretation

Bias, MAE, and RMSE pool all common finite spatial points across matched
forecast cases. Spatial correlation is the mean of per-case spatial Pearson
correlations. Positive improvement values mean the fine-tuned rollout is
better. Percentage reduction is
`100 * (baseline - fine-tuned) / abs(baseline)` and is undefined when the
absolute baseline is at or below `{float(EVAL['percentage_epsilon']):g}`.

## Largest improvements by variable and level

{markdown_table(largest, target_columns)}

## Variables and levels that became worse

A row appears here when mean RMSE reduction, mean MAE reduction, or mean
spatial-correlation change is negative.

{markdown_table(worse, target_columns)}

## Change with forecast lead time

{markdown_table(by_lead, lead_columns)}

## Missing variable/level/lead combinations

{markdown_table(missing, ['variable', 'level', 'lead_time_hours'])}

## Skipped selections

{skipped_text}

## Alignment and data-quality warnings

{warning_text}
"""


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_CSV = OUTPUT_DIR / "evaluation_metrics.csv"
METRICS_NC = OUTPUT_DIR / "evaluation_metrics.nc"
SUMMARY_MD = OUTPUT_DIR / "evaluation_summary.md"
PER_CASE_CSV = OUTPUT_DIR / "evaluation_metrics_per_case.csv"

METRICS_DF.to_csv(METRICS_CSV, index=False)
dataframe_to_netcdf(METRICS_DF, METRICS_NC)
SUMMARY_MD.write_text(build_summary(METRICS_DF))
if EVAL["save_per_case_metrics"]:
    PER_CASE_DF.to_csv(PER_CASE_CSV, index=False)

print(f"Saved {METRICS_CSV}")
print(f"Saved {METRICS_NC}")
print(f"Saved {SUMMARY_MD}")
if EVAL["save_per_case_metrics"]:
    print(f"Saved {PER_CASE_CSV}")


Saved /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_metrics.csv
Saved /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_metrics.nc
Saved /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_summary.md
Saved /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_metrics_per_case.csv


## 6. Lead-time comparison figures

Five file-safe figures are produced for every evaluated variable and level:
RMSE, MAE, bias, spatial correlation, and a two-panel improvement view.


In [11]:
COMPARISON_FIGURES_REQUESTED = bool(
    EVAL.get("generate_comparison_figures", True)
)
MATPLOTLIB_REQUESTED = (
    COMPARISON_FIGURES_REQUESTED or bool(EVAL.get("generate_maps", False))
)
MATPLOTLIB_AVAILABLE = False
if MATPLOTLIB_REQUESTED:
    try:
        import matplotlib.pyplot as plt
        MATPLOTLIB_AVAILABLE = True
    except ImportError:
        message = (
            "Requested visualizations were skipped because matplotlib is not "
            "installed in this kernel. Metric artifacts remain complete."
        )
        METRIC_WARNINGS.add(message)
        warnings.warn(message, stacklevel=2)

PLOTTING_AVAILABLE = COMPARISON_FIGURES_REQUESTED and MATPLOTLIB_AVAILABLE


def file_safe(value: Any) -> str:
    """Convert a label to a conservative file-safe token."""
    token = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip())
    return token.strip("._-") or "unnamed"


def title_level(level: str, level_units: str = "") -> str:
    """Format a level for plot titles."""
    if level == "surface":
        return "surface"
    return f"{level} {level_units}".strip()


def plot_pair(
    subset: pd.DataFrame,
    baseline_column: str,
    finetuned_column: str,
    metric_name: str,
    ylabel: str,
    output_path: Path,
) -> None:
    """Plot baseline and fine-tuned metric curves versus lead time."""
    subset = subset.sort_values("lead_time_hours")
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.plot(
        subset["lead_time_hours"], subset[baseline_column],
        marker="o", linewidth=2, label="Aurora"
    )
    ax.plot(
        subset["lead_time_hours"], subset[finetuned_column],
        marker="o", linewidth=2, label="Fine-tuned"
    )
    variable = subset["variable"].iloc[0]
    level = subset["level"].iloc[0]
    level_units = str(subset["level_units"].iloc[0]).strip()
    ax.set_title(f"{metric_name}: {variable} ({title_level(level, level_units)})", fontsize=14)
    ax.set_xlabel("Forecast lead time (hours)")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def plot_improvements(subset: pd.DataFrame, output_path: Path) -> None:
    """Plot percentage error reductions and correlation change."""
    subset = subset.sort_values("lead_time_hours")
    fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(7.5, 7.2), sharex=True)
    for column, label in (
        ("bias_improvement_percent", "|bias| reduction"),
        ("mae_improvement_percent", "MAE reduction"),
        ("rmse_improvement_percent", "RMSE reduction"),
    ):
        ax_top.plot(
            subset["lead_time_hours"], subset[column], marker="o", linewidth=2,
            label=label,
        )
    ax_top.axhline(0, color="black", linewidth=1)
    ax_top.set_title("Bias, MAE, and RMSE reductions")
    ax_top.set_ylabel("Reduction relative to baseline (%)")
    ax_top.grid(True, alpha=0.3)
    ax_top.legend()

    ax_bottom.plot(
        subset["lead_time_hours"],
        subset["spatial_correlation_improvement"],
        marker="o", linewidth=2, color="tab:purple",
        label="Fine-tuned − baseline correlation",
    )
    ax_bottom.axhline(0, color="black", linewidth=1)
    ax_bottom.set_title("Spatial correlation improvement")
    ax_bottom.set_xlabel("Forecast lead time (hours)")
    ax_bottom.set_ylabel("Spatial correlation change")
    ax_bottom.grid(True, alpha=0.3)
    ax_bottom.legend()

    variable = subset["variable"].iloc[0]
    level = subset["level"].iloc[0]
    level_units = str(subset["level_units"].iloc[0]).strip()
    fig.suptitle(
        f"Fine-tuned improvement: {variable} ({title_level(level, level_units)})"
    )
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


GENERATED_FIGURES: list[Path] = []
if PLOTTING_AVAILABLE:
    for (variable, level), subset in METRICS_DF.groupby(["variable", "level"], sort=True):
        subset = subset[subset["number_of_forecasts"] > 0]
        if subset.empty:
            continue
        units = str(subset["variable_units"].iloc[0]).strip()
        unit_suffix = f" ({units})" if units else ""
        stem = f"{file_safe(variable)}_{file_safe(level)}"
        definitions = [
            ("baseline_rmse", "finetuned_rmse", "RMSE", f"RMSE{unit_suffix}", "rmse"),
            ("baseline_mae", "finetuned_mae", "MAE", f"MAE{unit_suffix}", "mae"),
            ("baseline_bias", "finetuned_bias", "Bias", f"Mean bias{unit_suffix}", "bias"),
            (
                "baseline_spatial_correlation",
                "finetuned_spatial_correlation",
                "Spatial correlation",
                "Mean spatial correlation",
                "spatial_correlation",
            ),
        ]
        for base_col, fine_col, name, ylabel, suffix in definitions:
            path = FIGURE_DIR / f"{stem}_{suffix}.png"
            plot_pair(subset, base_col, fine_col, name, ylabel, path)
            GENERATED_FIGURES.append(path)
        improvement_path = FIGURE_DIR / f"{stem}_improvement.png"
        plot_improvements(subset, improvement_path)
        GENERATED_FIGURES.append(improvement_path)

    print(f"Saved {len(GENERATED_FIGURES)} lead-time comparison figures to {FIGURE_DIR}")
else:
    print("Lead-time comparison figures disabled or unavailable")


Saved 20 lead-time comparison figures to /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/figures


## 7. Optional spatial maps

Maps are generated only when `evaluation.generate_maps: true`. By default,
one common initialization is mapped for every selected
variable/level/lead. Use `map_initialization_times` and
`max_map_forecasts_per_selection` to control cost. The absolute-error
difference panel is `|fine-tuned error| - |baseline error|`; negative values
indicate improvement.


In [12]:
def selection_allows_variable(binding: VariableBinding) -> bool:
    """Return whether a variable passes optional map selection."""
    selected = EVAL.get("map_variables")
    if selected is None:
        return True
    names = {
        item if isinstance(item, str)
        else item.get("dataset_name") or item.get("name")
        for item in selected
    }
    return binding.spec.dataset_name in names


def selected_map_levels(binding: VariableBinding) -> list[float | None]:
    """Resolve optional map-level restrictions."""
    available = LEVELS_BY_VARIABLE[binding.spec.dataset_name]
    selected = EVAL.get("map_levels")
    if selected is None:
        return available
    if isinstance(selected, Mapping):
        selected = selected.get(binding.spec.dataset_name)
        if selected is None:
            return available
    wanted = [float(value) for value in selected]
    tolerance = float(EVAL["level_tolerance"])
    return [
        level for level in available
        if level is not None
        and any(np.isclose(level, value, atol=tolerance, rtol=0) for value in wanted)
    ] + ([None] if None in available and any(str(value).lower() == "surface" for value in selected) else [])


def requested_map_initializations() -> set[int] | None:
    """Parse optional initialization-time selection to nanosecond integer keys."""
    selected = EVAL.get("map_initialization_times")
    if selected is None:
        return None
    return {
        int(parse_datetime(value, context="map_initialization_times").astype(np.int64))
        for value in selected
    }


def robust_limits(values: Sequence[np.ndarray]) -> tuple[float, float]:
    """Return robust shared color limits over finite arrays."""
    finite = np.concatenate([value[np.isfinite(value)].reshape(-1) for value in values])
    if finite.size == 0:
        return 0.0, 1.0
    low, high = np.nanpercentile(finite, [1, 99])
    if not np.isfinite(low) or not np.isfinite(high) or low == high:
        low, high = float(np.nanmin(finite)), float(np.nanmax(finite))
    if low == high:
        high = low + 1.0
    return float(low), float(high)


def plot_spatial_map(
    truth: xr.DataArray,
    baseline: xr.DataArray,
    finetuned: xr.DataArray,
    *,
    variable: str,
    level: str,
    level_units: str,
    lead: float,
    initialization: np.datetime64,
    units: str,
    path: Path,
) -> None:
    """Plot truth, products, errors, and absolute-error difference."""
    obs = np.asarray(truth.values)
    base = np.asarray(baseline.values)
    fine = np.asarray(finetuned.values)
    base_error = base - obs
    fine_error = fine - obs
    abs_difference = np.abs(fine_error) - np.abs(base_error)
    field_limits = robust_limits([obs, base, fine])
    error_max = float(
        np.nanpercentile(
            np.abs(np.concatenate([base_error.reshape(-1), fine_error.reshape(-1)])),
            99,
        )
    )
    diff_max = float(np.nanpercentile(np.abs(abs_difference), 99))
    error_max = error_max if np.isfinite(error_max) and error_max > 0 else 1.0
    diff_max = diff_max if np.isfinite(diff_max) and diff_max > 0 else 1.0

    panels = [
        ("CAMS", obs, "viridis", field_limits),
        ("Aurora rollout", base, "viridis", field_limits),
        ("Fine-tuned rollout", fine, "viridis", field_limits),
        ("Aurora error (Aurora − CAMS)", base_error, "RdBu_r", (-error_max, error_max)),
        ("Fine-tuned error (fine-tuned − CAMS)", fine_error, "RdBu_r", (-error_max, error_max)),
        (
            "|fine-tuned error| − |Aurora error|",
            abs_difference,
            "RdBu_r",
            (-diff_max, diff_max),
        ),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    for ax, (title, values, cmap, limits) in zip(axes.flat, panels):
        mesh = ax.pcolormesh(
            truth.longitude.values,
            truth.latitude.values,
            values,
            shading="auto",
            cmap=cmap,
            vmin=limits[0],
            vmax=limits[1],
        )
        fig.colorbar(mesh, ax=ax, shrink=0.82, label=units or None)
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    fig.suptitle(
        f"{variable} ({title_level(level, level_units)}), "
        f"init {initialization}, lead {lead:g} h"
    )
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


GENERATED_MAPS: list[Path] = []
if EVAL["generate_maps"] and MATPLOTLIB_AVAILABLE:
    allowed_inits = requested_map_initializations()
    map_leads = [float(value) for value in EVAL["map_lead_times_hours"]]
    max_cases = max(1, int(EVAL["max_map_forecasts_per_selection"]))
    with DatasetCache(max_open=8) as baseline_cache, DatasetCache(max_open=8) as finetuned_cache:
        map_tasks = [
            (binding, level, lead)
            for binding in BINDINGS
            if selection_allows_variable(binding)
            for level in selected_map_levels(binding)
            for lead in map_leads
        ]
        for binding, level, lead in tqdm(map_tasks, desc="Generate maps", unit="selection"):
            base_records = record_lookup(BASELINE, lead)
            fine_records = record_lookup(FINETUNED, lead)
            common = sorted(set(base_records) & set(fine_records))
            if allowed_inits is not None:
                common = [value for value in common if value in allowed_inits]
            for init_ns in common[:max_cases]:
                expected = expected_valid_time(init_ns, lead)
                truth_index = truth_index_for_time(expected)
                if truth_index is None:
                    continue
                context = (
                    f"map/{binding.spec.dataset_name}/{level_label(level)}/"
                    f"init={np.datetime64(init_ns, 'ns')}/lead={lead:g}h"
                )
                try:
                    truth = extract_truth_field(
                        binding.truth_name, truth_index, level, context=context
                    )
                    baseline = extract_forecast_field(
                        BASELINE, base_records[init_ns], binding.baseline_name,
                        level, baseline_cache, context=context
                    )
                    finetuned = extract_forecast_field(
                        FINETUNED, fine_records[init_ns], binding.finetuned_name,
                        level, finetuned_cache, context=context
                    )
                    truth, baseline, finetuned = align_fields(
                        truth, baseline, finetuned, context=context
                    )
                except EvaluationError as exc:
                    METRIC_WARNINGS.add(f"Map skipped: {exc}")
                    continue
                init_tag = pd.Timestamp(np.datetime64(init_ns, "ns")).strftime("%Y%m%dT%H%M%S")
                path = FIGURE_DIR / (
                    f"{file_safe(binding.spec.dataset_name)}_{file_safe(level_label(level))}_"
                    f"lead_{lead:g}h_init_{init_tag}_map.png"
                )
                plot_spatial_map(
                    truth, baseline, finetuned,
                    variable=binding.spec.dataset_name,
                    level=level_label(level),
                    level_units=LEVEL_UNITS_BY_VARIABLE[binding.spec.dataset_name],
                    lead=lead,
                    initialization=np.datetime64(init_ns, "ns"),
                    units=binding.units,
                    path=path,
                )
                GENERATED_MAPS.append(path)
    print(f"Saved {len(GENERATED_MAPS)} optional maps")
elif EVAL["generate_maps"]:
    print("Spatial maps skipped because matplotlib is unavailable")
else:
    print("Spatial maps disabled (evaluation.generate_maps: false)")


Spatial maps disabled (evaluation.generate_maps: false)


## 8. Overall aggregate spatial maps

When `evaluation.generate_overall_maps: true`, this section streams every
matched initialization and requested lead into one spatial comparison per
target variable and level. The five panels show mean CAMS, mean Aurora,
mean fine-tuned output, Aurora bias, and fine-tuned bias. Bias is the mean
signed forecast-minus-CAMS residual, so it can be negative; MAE cannot.
The two bias maps are centered below the gaps in the top row. The three top
maps share one horizontal colorbar between the rows, and the two bias maps
share one horizontal colorbar below the bottom row. Every panel uses a
Cartopy Plate Carree map background with land, ocean, coastlines, borders, and geographic
gridlines. Aggregate fields are also saved to
`overall_spatial_differences.nc`.


In [13]:
GENERATED_OVERALL_MAPS: list[Path] = []
OVERALL_SPATIAL_NC = OUTPUT_DIR / "overall_spatial_differences.nc"
if EVAL.get("generate_overall_maps", False):
    from finetune.generate_overall_evaluation_maps import generate_overall_maps

    GENERATED_OVERALL_MAPS = generate_overall_maps(CONFIG_PATH)
    print(f"Saved {len(GENERATED_OVERALL_MAPS)} overall aggregate maps")
else:
    print("Overall aggregate maps disabled (evaluation.generate_overall_maps: false)")


Aggregate overall maps:   0%|          | 0/184 [00:00<?, ?initialization/s]

Aggregate overall maps:   1%|          | 2/184 [00:00<00:11, 15.40initialization/s]

Aggregate overall maps:   2%|▏         | 4/184 [00:00<00:11, 15.44initialization/s]

Aggregate overall maps:   3%|▎         | 6/184 [00:00<00:11, 15.42initialization/s]

Aggregate overall maps:   4%|▍         | 8/184 [00:00<00:11, 15.45initialization/s]

...

Aggregate overall maps:  98%|█████████▊| 180/184 [00:11<00:00, 16.37initialization/s]

Aggregate overall maps: 100%|██████████| 184/184 [00:11<00:00, 21.89initialization/s]

Aggregate overall maps: 100%|██████████| 184/184 [00:11<00:00, 15.70initialization/s]

Skipped 21 requested forecast steps outside the CAMS truth valid-time range; no nearest-time matching was attempted.


Saved aggregate spatial data: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/overall_spatial_differences.nc
Saved overall map: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/figures/overall/no2_1000_overall_differences.png
Saved overall map: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/figures/overall/no2_925_overall_differences.png
Saved overall map: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/figures/overall/no2_850_overall_differences.png
Saved overall map: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/figures/overall/tcno2_surface_overall_differences.png
Saved 4 overall aggregate maps


## 9. Final validation

This cell confirms required artifacts, figure coverage, optional detailed
output, and all skipped variable/level/lead selections. It also closes the
ground-truth dataset without modifying any input NetCDF.


In [14]:
# Refresh the summary after optional map generation so late warnings are included.
SUMMARY_MD.write_text(build_summary(METRICS_DF))

REQUIRED_OUTPUTS = [METRICS_CSV, METRICS_NC, SUMMARY_MD]
missing_outputs = [path for path in REQUIRED_OUTPUTS if not path.is_file()]
empty_outputs = [path for path in REQUIRED_OUTPUTS if path.is_file() and path.stat().st_size == 0]
if missing_outputs or empty_outputs:
    raise RuntimeError(
        f"Output validation failed. Missing={missing_outputs}, empty={empty_outputs}"
    )

if PLOTTING_AVAILABLE:
    expected_figure_groups = int(
        (METRICS_DF[METRICS_DF["number_of_forecasts"] > 0][["variable", "level"]])
        .drop_duplicates()
        .shape[0]
    )
    expected_comparison_figures = expected_figure_groups * 5
    if len(GENERATED_FIGURES) != expected_comparison_figures:
        raise RuntimeError(
            f"Expected {expected_comparison_figures} comparison figures, "
            f"created {len(GENERATED_FIGURES)}."
        )

if EVAL["save_per_case_metrics"] and not PER_CASE_CSV.is_file():
    raise RuntimeError(f"Expected per-case metrics output was not created: {PER_CASE_CSV}")

if EVAL.get("generate_overall_maps", False):
    expected_overall_maps = sum(len(values) for values in LEVELS_BY_VARIABLE.values())
    if len(GENERATED_OVERALL_MAPS) != expected_overall_maps:
        raise RuntimeError(
            f"Expected {expected_overall_maps} overall maps, "
            f"created {len(GENERATED_OVERALL_MAPS)}."
        )
    if not OVERALL_SPATIAL_NC.is_file():
        raise RuntimeError(
            f"Expected aggregate spatial NetCDF was not created: {OVERALL_SPATIAL_NC}"
        )

TRUTH_DS.close()

print("Evaluation output validation passed.")
for path in REQUIRED_OUTPUTS:
    print(f"  {path} ({path.stat().st_size:,} bytes)")
print(f"  comparison figures: {len(GENERATED_FIGURES)}")
print(f"  optional spatial maps: {len(GENERATED_MAPS)}")
print(f"  overall aggregate maps: {len(GENERATED_OVERALL_MAPS)}")
if EVAL["save_per_case_metrics"]:
    print(f"  per-case metrics: {PER_CASE_CSV}")

if SKIPPED:
    print("\nSkipped variable/level/lead selections:")
    for item in SKIPPED:
        print(
            f"  {item['variable']} / {item['level']} / "
            f"{item['lead_time_hours']:g} h: {item['reason']}"
        )
else:
    print("\nNo variable/level/lead selections were skipped.")

all_warnings = DISCOVERY_WARNINGS.messages() + METRIC_WARNINGS.messages()
if all_warnings:
    print("\nAlignment/data-quality warnings (also recorded in evaluation_summary.md):")
    for message in all_warnings:
        print(f"  - {message}")
else:
    print("\nNo alignment or data-quality warnings.")


Evaluation output validation passed.
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_metrics.csv (12,343 bytes)
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_metrics.nc (34,174 bytes)
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_summary.md (8,203 bytes)
  comparison figures: 20
  optional spatial maps: 0
  overall aggregate maps: 4
  per-case metrics: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/evaluation/evaluation_metrics_per_case.csv

No variable/level/lead selections were skipped.

Alignment/data-quality warnings (also recorded in evaluation_summary.md):
  - Numeric lead coordinate `lead_time` has no units; interpreted as hours by evaluation.lead_time_numeric_units. (repeated 854 times)
  - fine-tuned skipped an unusable NetCDF file: rollout_predictions.nc: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching/rollout_p